# Enron local-data cleaning and assessment

Run all cells in order. Requires Python plus `numpy`, `pyarrow` and `tzdata` (`pip install numpy pyarrow tzdata`; `tzdata` supplies the timezone database Windows lacks). No Elasticsearch connection is needed. Source: the course-provided JSON archive, https://s3.amazonaws.com/inta.gatech/inta6450-emails.zip . Do not alter the extracted JSON files.

**Two stages.** Stage 0 reads the 251,734 small JSON files once into `enron_raw.parquet` (exact original bytes plus a SHA-256 hash for each file). Stage 1 cleans from that file, so changing a cleaning rule and rerunning does not re-read every file. Stage 0 is skipped on later runs unless the source file list changes or `REBUILD_RAW = True`.

This notebook creates a reproducible, retrieval-independent analysis table and a review sample. It does **not** classify wrongdoing or apply a date cutoff. Exploratory data analysis (EDA), lexical/semantic retrieval, near-duplicate clustering, and human Risk/Awareness/Action assessment follow this stage.

| File | Location | Role |
|---|---|---|
| `enron_raw.parquet` | project folder, reused | **Evidence archive.** Exact original files. Quote and verify final emails from here (`load_raw`). |
| `emails_analysis.parquet` | run folder | **Main table.** 18 columns, one row per email. Input for EDA, TF-IDF, Elasticsearch, near-duplicate clustering and context review. |
| `quality_report.json` | run folder | Run configuration, funnel counts, flag counts, source fingerprint, and any parse errors. |
| `cleaning_review_sample.jsonl` | run folder | 40 random emails (original and new text) for a manual check of the cleaning rules. |

Each run writes a new timestamped run folder under `enron_cleaning_outputs`. See `Enron_Data_Dictionary.md` for field definitions.

## 1. Configuration
Edit these settings before running. Defaults process the full corpus.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timedelta, timezone
from email.utils import parsedate_to_datetime, getaddresses
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
import hashlib
import json
import random
import re
import unicodedata
import platform
import zlib

try:
    import numpy as np
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError as error:
    raise ImportError('Install the required packages: pip install numpy pyarrow tzdata') from error
try:
    HOUSTON = ZoneInfo('America/Chicago')   # Enron HQ; used for the downstream cutoff helper
except ZoneInfoNotFoundError as error:
    raise ImportError('Timezone data is missing (common on Windows): pip install tzdata') from error

# --- Paths ---
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'inta6450-emails' / 'dataset'   # extracted course JSON files
OUTPUT_PARENT = PROJECT_ROOT / 'enron_cleaning_outputs'   # each run gets its own subfolder
RAW_PARQUET = PROJECT_ROOT / 'enron_raw.parquet'          # Stage 0 output, reused across runs

# --- Run settings ---
REBUILD_RAW = False        # True forces Stage 0 to re-read the JSON files (e.g. if source files changed)
MAX_FILES = None           # None = all files; e.g. 1000 for a quick pilot run (Stage 1 only)
RANDOM_SEED = 6450         # makes the review sample and MinHash functions reproducible
REVIEW_SAMPLE_SIZE = 40

# --- Duplicate / near-duplicate settings ---
DUP_MAX_SHIFT_HOURS = 26   # the same email in two mailboxes can carry dates shifted by whole timezone hours
SHINGLE_SIZE = 5           # words per shingle for MinHash
MINHASH_PERMUTATIONS = 64  # MinHash signature length

# --- Validate settings early so a long run doesn't fail halfway ---
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'Expected {DATA_DIR}; set PROJECT_ROOT to your project folder.')
if MAX_FILES is not None and (not isinstance(MAX_FILES, int) or MAX_FILES <= 0):
    raise ValueError('MAX_FILES must be None or a positive integer.')
print('Data folder:', DATA_DIR)

Data folder: D:\E_Guo\z_New_folder\INTA6450\inta6450-emails\dataset


## 2. Cleaning rules

- Each filename stem is a local `email_id`; it is **not assumed** to be the hosted Elasticsearch `_id`.
- **Dates.** Parse the structured date and the original header date independently. The structured date is preferred; the header date is a fallback (flag `date_from_header`). Naive, missing, and invalid dates are flagged rather than guessed (`date_unresolved` if neither parses). Disagreement between the two is flagged. Dates outside 1990–2005 are inspection flags, not exclusions.
- **No cutoff here.** The scope cutoff is applied downstream with `cutoff_utc()` (Section 7), so changing it never requires rerunning the cleaning.
- **Participants.** Use structured address lists, falling back to original headers only when absent. Lowercase addresses. No speculative person-alias merging. Address lists are always lists (possibly empty), never null. `custodian` and `folder` record whose mailbox (and which folder) the copy was collected from, which can differ from the addressees.
- **Text.** `body_clean` = original text with Unicode normalized to NFC, one line-ending style, and repeated spaces/tabs collapsed; case, punctuation, numbers, signatures and quoted text are kept.
- **New vs. quoted text.** `body_new_candidate` cuts `body_clean` at the earliest recognized start of quoted/forwarded material: Outlook "Original Message", Lotus Notes "Forwarded by", "Forwarded message", "On … wrote:", an Outlook `From:`/`Sent:` header block, or a Lotus Notes `To:`/`cc:`/`Subject:` block (plus the sender/date lines above it). Lines starting with `>` are also removed (flag `inline_quotes_removed`). It is a heuristic, not certified author-only text. `quote_marker_type` records which rule fired.
- **Duplicate copies.** A later record is a copy of an earlier one when sender, subject and full cleaned body are identical and the dates are equal or differ by a whole number of half-hours up to `DUP_MAX_SHIFT_HOURS` (flag `duplicate_timezone_shift`). Copies point to the first copy via `canonical_email_id`; they are never deleted.
- **Near duplicates.** A MinHash signature of lowercase word 5-shingles of `body_clean` is stored for each email with at least 5 words. LSH clustering and its similarity threshold are chosen and validated downstream.
- No wrongdoing-term or date exclusions during cleaning.

**Quality flags** mark records that need a closer look. They do not delete anything. Their counts tell the EDA stage where problems are concentrated; whether a problem matters is decided by inspecting examples.

## 3. Helper functions
Small functions used by the main cleaning function, then a unit test cell.

In [2]:
def normalize_text(value):
    """Unicode NFC, unify line endings, collapse spaces/tabs; keeps case, punctuation and paragraphs."""
    value = unicodedata.normalize('NFC', value).replace('\r\n', '\n').replace('\r', '\n')
    return '\n'.join(re.sub(r'[ \t]+', ' ', line).rstrip() for line in value.split('\n')).strip()


def parse_date(value, header=False):
    """Return (UTC datetime or None, status). header=True parses RFC-2822 email header dates."""
    if value is None or value == '':
        return None, 'missing'
    if not isinstance(value, str):
        return None, 'invalid_type'
    try:
        result = parsedate_to_datetime(value) if header else datetime.fromisoformat(value.replace('Z', '+00:00'))
    except (ValueError, TypeError, OverflowError):
        return None, 'invalid'
    if result.tzinfo is None:          # never guess a timezone
        return None, 'timezone_missing'
    return result.astimezone(timezone.utc), 'ok'


# Mentions of attachments (the attachment files themselves are not in the corpus)
ATTACHMENT_HINT = re.compile(r'(?i)\battach(?:ed|ment|ments)\b|\.(?:xls[x]?|doc[x]?|pdf|ppt[x]?)\b')

In [3]:
# Patterns that mark the START of quoted or forwarded material. The earliest match in an email wins.
# Approximate counts of emails containing each pattern in this corpus guided the list:
#   Outlook "-----Original Message-----"          ~57k  (also "Original Appointment", sometimes mid-line)
#   Lotus Notes "----- Forwarded by ..." (dashes)  ~36k  (the old rule missed the leading dashes)
#   Outlook "From: ... / Sent: ..." header block   ~54k  (often without an Original Message line)
#   Lotus Notes "To: ... / cc: ... / Subject:"     common in Notes replies
#   "On <date>, <person> wrote:"                   ~140  (rare in Enron's mail clients)
QUOTE_PATTERNS = [
    ('outlook_original_message', r'-{2,}[ \t]*original (?:message|appointment)[ \t]*-{2,}'),   # may follow text on the same line
    ('forwarded_by', r'(?:^[ \t]*|-{2,}[ \t]*)forwarded by\b'),
    ('forwarded_message', r'^[ \t]*(?:-+[ \t]*forwarded message[ \t]*-+|begin forwarded message:)[ \t]*$'),
    ('on_wrote', r'^[ \t]*on\b[^\n]{0,200}\bwrote:[ \t]*$'),
    ('outlook_header_block', r'^[ \t]*from:[^\n]*\n[ \t]*sent:'),
    ('notes_header_block', r'^[ \t]*to:[^\n]*\n(?:[^\n]*\n){0,10}?[ \t]*(?:cc|subject):'),   # To: lists can wrap over many lines
]
QUOTE_REGEXES = [(name, re.compile(pattern, re.IGNORECASE | re.MULTILINE)) for name, pattern in QUOTE_PATTERNS]

# Lines just above a Lotus Notes "To:" block usually hold the original sender and date, e.g.
#   John Doe@ECT
#   10/23/2000 09:12 AM
NOTES_PREAMBLE = re.compile(r'@|\b\d{1,2}/\d{1,2}/\d{2,4}\b|\b\d{1,2}:\d{2}\b')
INLINE_QUOTE_LINE = re.compile(r'^[ \t]*>')


def extend_notes_preamble(body, start, max_lines=3, max_blank=2):
    """Move the cut point up over short sender/date lines above a Notes To: block.
    Up to `max_blank` blank lines may separate them from the To: line."""
    position, taken, blanks = start, 0, 0
    while position > 0 and taken < max_lines:
        previous_start = body.rfind('\n', 0, position - 1) + 1
        line = body[previous_start:position - 1]
        if not line.strip():
            blanks += 1
            if blanks > max_blank:
                break
            position = previous_start
            continue
        if len(line.strip()) > 100 or not NOTES_PREAMBLE.search(line):
            break
        start = position = previous_start   # include this line (and blank lines below it) in the quote
        taken += 1
        blanks = 0
    return start


def find_quote_start(body):
    """Return (position, pattern name) of the earliest quote marker, or (None, None)."""
    best = None
    for name, regex in QUOTE_REGEXES:
        match = regex.search(body)
        if match and (best is None or match.start() < best[0]):
            best = (match.start(), name)
    if best is None:
        return None, None
    start, name = best
    if name == 'notes_header_block':
        start = extend_notes_preamble(body, start)
    return start, name


def split_quoted(body):
    """Return (new_text, quoted_text, marker_type, inline_lines_removed).
    New text = text before the first marker, minus any '>'-prefixed inline-reply lines."""
    start, name = find_quote_start(body)
    head = body if start is None else body[:start]
    quoted = '' if start is None else body[start:]
    kept, removed = [], 0
    for line in head.split('\n'):
        if INLINE_QUOTE_LINE.match(line):
            removed += 1
        else:
            kept.append(line)
    return '\n'.join(kept).strip(), quoted, name, removed

In [4]:
def addresses(record, headers, field):
    """Return (sorted unique lowercase addresses, flags) for from/to/cc/bcc. Always a list, never None.
    Prefers the structured list; falls back to the raw header only when the field is absent."""
    flags = []
    value = record.get(field)
    if value is None:
        raw = headers.get(field, '')
        if not isinstance(raw, str):
            return [], [f'{field}_header_invalid_type']
        return sorted({a.strip().lower() for _, a in getaddresses([raw]) if a.strip()}), flags
    if not isinstance(value, list):
        return [], [f'{field}_invalid_type']
    found = []
    for item in value:
        if not isinstance(item, dict) or not isinstance(item.get('address'), str):
            flags.append(f'{field}_invalid_entry')
        elif item['address'].strip():
            found.append(item['address'].strip().lower())
    return sorted(set(found)), flags

In [5]:
# --- MinHash signature for downstream near-duplicate (LSH) clustering ---
TOKEN = re.compile(r'[a-z0-9]+')
_minhash_rng = np.random.default_rng(RANDOM_SEED)
MINHASH_PRIME = np.uint64(4294967311)   # prime just above 2**32
MINHASH_A = _minhash_rng.integers(1, 2**31, size=MINHASH_PERMUTATIONS, dtype=np.uint64)
MINHASH_B = _minhash_rng.integers(0, 2**31, size=MINHASH_PERMUTATIONS, dtype=np.uint64)


def minhash_signature(text, chunk=20000):
    """MinHash of word shingles. Lowercase letters/digits only, so whitespace, punctuation and
    line-wrap differences do not matter. Returns None if the text has fewer than SHINGLE_SIZE words."""
    tokens = TOKEN.findall(text.casefold())
    if len(tokens) < SHINGLE_SIZE:
        return None
    shingles = {' '.join(tokens[i:i + SHINGLE_SIZE]) for i in range(len(tokens) - SHINGLE_SIZE + 1)}
    hashes = np.fromiter((zlib.crc32(s.encode('utf-8')) for s in shingles), dtype=np.uint64, count=len(shingles))
    signature = np.full(MINHASH_PERMUTATIONS, np.iinfo(np.uint64).max, dtype=np.uint64)
    for i in range(0, len(hashes), chunk):   # chunks keep memory bounded for very long emails
        block = (hashes[i:i + chunk, None] * MINHASH_A + MINHASH_B) % MINHASH_PRIME
        signature = np.minimum(signature, block.min(axis=0))
    return signature.tolist()


def minhash_similarity(a, b):
    """Estimated Jaccard similarity: share of positions where two signatures agree."""
    return sum(x == y for x, y in zip(a, b)) / len(a)


# --- Duplicate copies: same content, dates equal or shifted by whole half-hours ---
def match_duplicate(index, key, date, email_id):
    """Return (canonical_email_id, reason). `index` maps content key -> [(date, email_id)] of
    canonical copies seen so far. reason is 'identical', 'timezone_shift' or 'none'."""
    if key is None or date is None:
        return email_id, 'none'
    copies = index.setdefault(key, [])
    for canonical_date, canonical_id in copies:
        delta = abs((date - canonical_date).total_seconds())
        if delta == 0:
            return canonical_id, 'identical'
        if delta <= DUP_MAX_SHIFT_HOURS * 3600 and delta % 1800 == 0:
            return canonical_id, 'timezone_shift'
    copies.append((date, email_id))
    return email_id, 'none'

In [6]:
def clean_record(record, email_id):
    """Turn one raw JSON email into one cleaned row (dict). Raises ValueError for unusable records.
    Keys starting with '_' are used internally (duplicate matching, review sample) and are not saved."""
    # --- Basic shape checks: bad records are quarantined by the caller ---
    if not isinstance(record, dict):
        raise ValueError('Top-level JSON must be an object.')
    raw = record.get('text')
    if not isinstance(raw, str):
        raise ValueError('text must be a string; record requires manual review.')
    headers = record.get('headers', {})
    if not isinstance(headers, dict):
        raise ValueError('headers must be an object.')
    flags = []

    # --- Body: normalize, then split new text from quoted/forwarded text ---
    body = normalize_text(raw)
    new, _quoted, marker_type, inline_removed = split_quoted(body)
    if inline_removed:
        flags.append('inline_quotes_removed')

    # --- Dates: structured field preferred, header as fallback, disagreement flagged ---
    structured_date, structured_status = parse_date(record.get('date'))
    header_date, header_status = parse_date(headers.get('date'), header=True)
    date = structured_date if structured_date is not None else header_date
    if structured_status != 'ok':
        flags.append('structured_date_' + structured_status)
    if header_status != 'ok':
        flags.append('header_date_' + header_status)
    if structured_date is None and header_date is not None:
        flags.append('date_from_header')
    if date is None:
        flags.append('date_unresolved')
    if structured_date is not None and header_date is not None and structured_date != header_date:
        flags.append('date_disagreement')
    if date is not None and not 1990 <= date.year <= 2005:
        flags.append('date_outside_inspection_range')

    # --- Participants (always lists, possibly empty) ---
    address_fields = {}
    for source, target in [('from', 'sender'), ('to', 'recipients'), ('cc', 'cc'), ('bcc', 'bcc')]:
        address_fields[target], issues = addresses(record, headers, source)
        flags.extend(issues)
    if not address_fields['sender']:
        flags.append('missing_sender')
    if not (address_fields['recipients'] or address_fields['cc'] or address_fields['bcc']):
        flags.append('missing_recipients')

    # --- Subject: strip leading RE:/FW: prefixes for grouping hints ---
    subject = record.get('subject', headers.get('subject', ''))
    if not isinstance(subject, str):
        flags.append('subject_invalid_type')
        subject = ''
    subject_norm = re.sub(r'(?i)^(?:(?:re|fw|fwd)\s*:\s*)+', '', normalize_text(subject)).casefold()

    # --- Content quality flags ---
    if not body:
        flags.append('empty_body')
    if len(new) < 20:
        flags.append('short_new_text')
    if marker_type is not None and not new:
        flags.append('quote_split_at_start')     # marker at the very top: check the split is right
    if ATTACHMENT_HINT.search(raw):
        flags.append('attachment_mentioned')     # attachment files themselves are not in the corpus
    if '\ufffd' in raw:
        flags.append('unicode_replacement_character')
    signature = minhash_signature(body)
    if signature is None:
        flags.append('too_short_for_minhash')

    # --- Content key for duplicate copies (date is compared separately, allowing timezone shifts) ---
    content_key = None
    if address_fields['sender'] and body:
        identity = [address_fields['sender'], normalize_text(subject), body]
        content_key = hashlib.sha256(json.dumps(identity, ensure_ascii=False).encode('utf-8')).hexdigest()

    return {
        'email_id': email_id,
        'message_id': record.get('messageId') or headers.get('message-id'),
        'custodian': headers.get('x-origin'), 'folder': headers.get('x-folder'),
        'date_utc': date,
        **address_fields,
        'subject': subject, 'subject_norm': subject_norm,
        'body_clean': body, 'body_new_candidate': new, 'quote_marker_type': marker_type,
        'minhash_signature': signature,
        'quality_flags': sorted(set(flags)),
        '_content_key': content_key, '_body_raw': raw,
    }

In [7]:
# Quick checks of the rules above on hand-made examples (no real data needed)
assert normalize_text('e\u0301\r\n a\t b ') == '\u00e9\n a b'
assert parse_date('2001-11-08T19:24:50.000Z')[0].isoformat() == '2001-11-08T19:24:50+00:00'
assert parse_date('2001-11-08')[1] == 'timezone_missing'
assert parse_date('not a date')[1] == 'invalid'
assert parse_date('Thu, 8 Nov 2001 11:24:50 -0800 (PST)', header=True)[0] == parse_date('2001-11-08T19:24:50Z')[0]

# Quote splitting: one example per pattern
examples = {
    'outlook_original_message': 'Do not approve.\n -----Original Message-----\nFrom: Someone\nPrior evidence',
    'forwarded_by': 'FYI\n---------------------- Forwarded by Vince J Kaminski/HOU/ECT on 10/04/2001 08:30 AM ---------------------------\nPrior evidence',
    'on_wrote': 'Agreed.\nOn Thu, 14 Sep 2000 John.Arnold@enron.com wrote:\nPrior evidence',
    'outlook_header_block': 'See below.\nFrom: Erb, Kim [mailto:KErb@x.com]\nSent: Monday, May 14, 2001 9:00 AM\nPrior evidence',
    'notes_header_block': 'Thanks.\n\nJohn Doe@ECT\n10/23/2000 09:12 AM\nTo: Jane Smith/HOU/ECT@ECT\ncc:\nSubject: Re: Raptor\nPrior evidence',
}
for expected, text in examples.items():
    new, quoted, marker, _ = split_quoted(normalize_text(text))
    assert marker == expected, (expected, marker)
    assert 'Prior evidence' in quoted and 'Prior evidence' not in new, expected
assert split_quoted(normalize_text(examples['notes_header_block']))[0] == 'Thanks.'           # sender/date lines moved to quoted
notes_gap = 'Call after T-giving. Sara\n\n "Virginia Hodge" <vh@x.com>\n 11/17/2000 10:49 AM\n\n To: sara@enron.com\n cc:\n Subject: Hi\nPrior'
assert split_quoted(normalize_text(notes_gap))[0] == 'Call after T-giving. Sara'          # blank line before To: block
assert split_quoted('fyi -----Original Message-----\nFrom: X\nSent: Y\nPrior')[:3] == ('fyi', '-----Original Message-----\nFrom: X\nSent: Y\nPrior', 'outlook_original_message')
wrapped = 'Any word from PJM?\n\nSusan J Mara@ENRON\n03/13/2001 03:06 PM\nTo: Richard Shapiro/NA/Enron@Enron, Joe\nHartsoe/Corp/Enron@ENRON, Jeff\nDasovich/NA/Enron@Enron, Mona\nPetrochko/NA/Enron@Enron\ncc:\nSubject: PJM\nPrior'
assert split_quoted(wrapped)[0] == 'Any word from PJM?'                                    # To: list wraps over lines
assert split_quoted('TO: AD HOC GROUP\n\nWe received good coverage today.')[2] is None     # memo-style To: is not a quote
assert split_quoted('Please place on my calendar.\n-----Original Appointment-----\nFrom: X')[2] == 'outlook_original_message'
assert split_quoted('My answer\n> quoted line\nmore answer') == ('My answer\nmore answer', '', None, 1)
assert split_quoted('Plain email with no quotes.') == ('Plain email with no quotes.', '', None, 0)

# Full record: lists never None, flags replace the removed columns
fixture = {'text': examples['outlook_original_message'] + '\nSee attached.', 'date': '2001-01-01T00:00:00Z',
           'from': [{'address': 'A@ENRON.COM'}], 'subject': 'RE: FW: Raptor'}
example = clean_record(fixture, 'test')
assert example['body_new_candidate'] == 'Do not approve.' and example['_body_raw'] == fixture['text']
assert example['subject_norm'] == 'raptor' and example['sender'] == ['a@enron.com']
assert example['recipients'] == [] and 'missing_recipients' in example['quality_flags']
assert 'attachment_mentioned' in example['quality_flags'] and example['quote_marker_type'] == 'outlook_original_message'
header_only = clean_record({**fixture, 'date': None, 'headers': {'date': 'Mon, 1 Jan 2001 00:00:00 +0000'}}, 'test')
assert 'date_from_header' in header_only['quality_flags'] and header_only['date_utc'] is not None
assert 'date_unresolved' in clean_record({**fixture, 'date': None}, 'test')['quality_flags']
assert 'inline_quotes_removed' in clean_record({**fixture, 'text': 'Mine\n> theirs'}, 'test')['quality_flags']

# Duplicate copies: identical date, whole-hour timezone shift, and a genuinely different time
index = {}
t0 = datetime(2002, 1, 25, 21, 37, 19, tzinfo=timezone.utc)
assert match_duplicate(index, 'k', t0, 'first') == ('first', 'none')
assert match_duplicate(index, 'k', t0, 'second') == ('first', 'identical')
assert match_duplicate(index, 'k', t0 + timedelta(hours=4), 'third') == ('first', 'timezone_shift')
assert match_duplicate(index, 'k', t0 + timedelta(minutes=7), 'fourth') == ('fourth', 'none')
assert match_duplicate(index, None, t0, 'fifth') == ('fifth', 'none')

# MinHash: whitespace/punctuation differences give identical signatures; unrelated text is dissimilar
base = 'The Raptor vehicles cannot absorb the loss on the hedged positions this quarter.'
assert minhash_signature(base) == minhash_signature(base.upper().replace(' ', '  \n') + '!!')
edited = base + ' Please review before the board meeting.'
assert minhash_similarity(minhash_signature(base), minhash_signature(edited)) > 0.3
assert minhash_similarity(minhash_signature(base), minhash_signature('Lunch menu for the Houston office party on Friday afternoon')) < 0.2
assert minhash_signature('too short') is None
print('Cleaning-rule tests passed.')

Cleaning-rule tests passed.


## 4. Stage 0: ingest raw files into Parquet (one time)
Reads each JSON file once and stores its exact bytes, so the evidence is unchanged and every later cleaning run reads one file instead of 251,734. The check on reruns compares the file list only; it does not re-hash file contents. If source files may have been modified, set `REBUILD_RAW = True`.

In [8]:
RAW_SCHEMA = pa.schema([
    ('email_id', pa.string()),        # filename stem
    ('source_file', pa.string()),     # project-relative path of the original file
    ('source_sha256', pa.string()),   # hash of the exact original bytes
    ('raw_bytes', pa.binary()),       # exact original file content, not decoded or altered
])


def build_raw_parquet(paths, target, batch_size=10000):
    """Copy every source file's bytes into one Parquet file. Writes to a temporary file first,
    so an interrupted run never leaves a half-written enron_raw.parquet behind."""
    partial = target.with_name(target.name + '.partial')
    buffer = []
    with pq.ParquetWriter(partial, RAW_SCHEMA, compression='zstd') as writer:
        for position, path in enumerate(paths, 1):
            payload = path.read_bytes()
            buffer.append({'email_id': path.stem, 'source_file': str(path.relative_to(PROJECT_ROOT)),
                           'source_sha256': hashlib.sha256(payload).hexdigest(), 'raw_bytes': payload})
            if len(buffer) >= batch_size:
                writer.write_table(pa.Table.from_pylist(buffer, schema=RAW_SCHEMA))
                buffer.clear()
            if position % 25000 == 0:
                print(f'Ingested {position:,}/{len(paths):,} files')
        if buffer:
            writer.write_table(pa.Table.from_pylist(buffer, schema=RAW_SCHEMA))
    partial.replace(target)


files = sorted(DATA_DIR.glob('*.json'))
if not files:
    raise FileNotFoundError('No JSON files found.')

if REBUILD_RAW or not RAW_PARQUET.exists():
    build_raw_parquet(files, RAW_PARQUET)
    print('Built', RAW_PARQUET)
else:
    # Reuse the existing copy only if it covers exactly the same files, in the same order
    stored_ids = pq.read_table(RAW_PARQUET, columns=['email_id']).column('email_id').to_pylist()
    if stored_ids != [p.stem for p in files]:
        raise RuntimeError('enron_raw.parquet does not match the current source files; set REBUILD_RAW = True.')
    print('Reusing', RAW_PARQUET)

raw_rows = pq.ParquetFile(RAW_PARQUET).metadata.num_rows
assert raw_rows == len(files)
print(f'{raw_rows:,} raw records available')

Reusing D:\E_Guo\z_New_folder\INTA6450\enron_raw.parquet
251,734 raw records available


## 5. Stage 1: clean every record
Streams the raw Parquet in batches: clean each record, flag duplicate copies, write the row, and keep a random review sample. Unreadable records are recorded rather than silently skipped. The original JSON files are not read in this stage.

In [9]:
selected_count = raw_rows if MAX_FILES is None else min(MAX_FILES, raw_rows)

# New timestamped folder per run, so earlier runs are never overwritten
run_name = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = OUTPUT_PARENT / run_name
RUN_DIR.mkdir(parents=True, exist_ok=False)

# Running tallies for the quality report
stats = Counter()
flag_counts = Counter()
marker_counts = Counter()
parse_errors = []                 # unreadable records (expected to be empty)
duplicate_index = {}              # content key -> [(date, email_id)] of canonical copies
rng = random.Random(RANDOM_SEED)
sample = []                       # reservoir sample for manual review
manifest_hash = hashlib.sha256()  # one fingerprint of all source files, for reproducibility


def iter_raw_records(limit, batch_size=5000):
    """Yield raw records from enron_raw.parquet in file order, stopping after `limit` records."""
    yielded = 0
    for batch in pq.ParquetFile(RAW_PARQUET).iter_batches(batch_size=batch_size):
        for item in batch.to_pylist():
            if yielded >= limit:
                return
            yielded += 1
            yield item


# Main table: 18 columns. Original text and headers stay in enron_raw.parquet (see load_raw).
# Address lists, flags and text fields are always present (possibly empty), never null,
# so downstream code can rely on lists and strings for these columns.
string_list = pa.list_(pa.string())
PARQUET_SCHEMA = pa.schema([
    pa.field('email_id', pa.string(), nullable=False),            # link to enron_raw.parquet
    ('message_id', pa.string()),
    pa.field('canonical_email_id', pa.string(), nullable=False),  # first copy of this email
    pa.field('is_duplicate_copy', pa.bool_(), nullable=False),    # canonical_email_id != email_id
    ('custodian', pa.string()), ('folder', pa.string()),          # whose mailbox / which folder
    ('date_utc', pa.timestamp('us', tz='UTC')),
    pa.field('sender', string_list, nullable=False), pa.field('recipients', string_list, nullable=False),
    pa.field('cc', string_list, nullable=False), pa.field('bcc', string_list, nullable=False),
    pa.field('subject', pa.string(), nullable=False), pa.field('subject_norm', pa.string(), nullable=False),
    pa.field('body_clean', pa.string(), nullable=False), pa.field('body_new_candidate', pa.string(), nullable=False),
    ('quote_marker_type', pa.string()),
    ('minhash_signature', pa.list_(pa.uint64())),
    pa.field('quality_flags', string_list, nullable=False),
])
NON_NULL_COLUMNS = [field.name for field in PARQUET_SCHEMA if not field.nullable]
PARQUET_BATCH_SIZE = 10000   # rows held in memory before each write


def to_parquet_row(row):
    """Pick the saved columns. pyarrow does not enforce nullable=False by itself, so it is checked here."""
    item = {name: row[name] for name in PARQUET_SCHEMA.names}
    missing = [name for name in NON_NULL_COLUMNS if item[name] is None]
    if missing:
        raise ValueError(f"{row['email_id']}: null value in non-nullable column(s) {missing}")
    return item


def flush_parquet(writer, buffer):
    """Write buffered rows to the Parquet file and empty the buffer."""
    if buffer:
        writer.write_table(pa.Table.from_pylist(buffer, schema=PARQUET_SCHEMA))
        buffer.clear()


# Schema check: a real cleaned row fits, and a None in a list column is rejected
_check = {**example, 'canonical_email_id': 'test', 'is_duplicate_copy': False}
pa.Table.from_pylist([to_parquet_row(_check)], schema=PARQUET_SCHEMA)
try:
    to_parquet_row({**_check, 'recipients': None})
    raise AssertionError('A null list column was accepted')
except ValueError:
    pass
print(f'Ready to clean {selected_count:,} records into {RUN_DIR}')

Ready to clean 251,734 records into D:\E_Guo\z_New_folder\INTA6450\enron_cleaning_outputs\20261111T025423476065Z


In [10]:
parquet_buffer = []
with pq.ParquetWriter(RUN_DIR / 'emails_analysis.parquet', PARQUET_SCHEMA, compression='zstd') as parquet_writer:
    for position, item in enumerate(iter_raw_records(selected_count), 1):
        # 1. Take the original bytes and hash stored by Stage 0
        manifest_hash.update(f"{item['source_file']}\t{item['source_sha256']}\n".encode('utf-8'))
        stats['files_attempted'] += 1

        # 2. Parse and clean; quarantine anything unreadable
        try:
            record = json.loads(item['raw_bytes'].decode('utf-8-sig'))
            row = clean_record(record, item['email_id'])
        except (UnicodeDecodeError, json.JSONDecodeError, ValueError) as error:
            stats['quarantined_records'] += 1
            parse_errors.append({'email_id': item['email_id'], 'source_file': item['source_file'],
                                 'error_type': type(error).__name__, 'error': str(error)})
            continue

        # 3. Duplicate copies: the first copy seen is canonical; later copies are flagged, not deleted
        canonical, reason = match_duplicate(duplicate_index, row['_content_key'], row['date_utc'], row['email_id'])
        row['canonical_email_id'] = canonical
        row['is_duplicate_copy'] = canonical != row['email_id']
        if reason == 'timezone_shift':
            row['quality_flags'] = sorted(row['quality_flags'] + ['duplicate_timezone_shift'])

        # 4. Update tallies
        stats['valid_records'] += 1
        stats['duplicate_copies'] += int(row['is_duplicate_copy'])
        stats['duplicate_copies_' + reason] += int(row['is_duplicate_copy'])
        stats['unique_content_representatives'] += int(not row['is_duplicate_copy'])
        marker_counts[row['quote_marker_type'] or 'none'] += 1
        flag_counts.update(row['quality_flags'])

        # 5. Buffer the row for the main table
        parquet_buffer.append(to_parquet_row(row))
        if len(parquet_buffer) >= PARQUET_BATCH_SIZE:
            flush_parquet(parquet_writer, parquet_buffer)

        # 6. Reservoir sampling: every record has an equal chance of being in the review sample
        sample_row = {'email_id': row['email_id'], 'subject': row['subject'],
                      'date_utc': row['date_utc'].isoformat() if row['date_utc'] else None,
                      'sender': row['sender'], 'custodian': row['custodian'], 'body_raw': row['_body_raw'],
                      'body_new_candidate': row['body_new_candidate'],
                      'quote_marker_type': row['quote_marker_type'], 'quality_flags': row['quality_flags']}
        if len(sample) < REVIEW_SAMPLE_SIZE:
            sample.append(sample_row)
        else:
            slot = rng.randrange(stats['valid_records'])
            if slot < REVIEW_SAMPLE_SIZE:
                sample[slot] = sample_row

        if position % 25000 == 0:
            print(f'Processed {position:,}/{selected_count:,} records')

    flush_parquet(parquet_writer, parquet_buffer)   # write the final partial batch

# Funnel must reconcile: nothing silently dropped
assert stats['files_attempted'] == selected_count
assert stats['files_attempted'] == stats['valid_records'] + stats['quarantined_records']
assert stats['valid_records'] == stats['duplicate_copies'] + stats['unique_content_representatives']
assert pq.ParquetFile(RUN_DIR / 'emails_analysis.parquet').metadata.num_rows == stats['valid_records']
print('Done:', dict(stats))
print('Quote markers:', dict(marker_counts))

Processed 25,000/251,734 records
Processed 50,000/251,734 records
Processed 75,000/251,734 records
Processed 100,000/251,734 records
Processed 125,000/251,734 records
Processed 150,000/251,734 records
Processed 175,000/251,734 records
Processed 200,000/251,734 records
Processed 225,000/251,734 records
Processed 250,000/251,734 records
Done: {'files_attempted': 251734, 'valid_records': 251734, 'duplicate_copies': 2491, 'duplicate_copies_none': 0, 'unique_content_representatives': 249243, 'duplicate_copies_timezone_shift': 2491}
Quote markers: {'outlook_original_message': 53419, 'forwarded_by': 29748, 'none': 142604, 'notes_header_block': 25785, 'outlook_header_block': 89, 'on_wrote': 70, 'forwarded_message': 19}


## 6. Write the run report and review sample

In [11]:
report = {
    'run_name': run_name,
    'status': 'needs_error_review' if parse_errors else 'processing_complete_review_pending',
    'python_version': platform.python_version(), 'pyarrow_version': pa.__version__, 'numpy_version': np.__version__,
    'source_url': 'https://s3.amazonaws.com/inta.gatech/inta6450-emails.zip',
    'available_json_files': len(files), 'selected_files': selected_count, 'is_full_corpus': selected_count == len(files),
    'raw_parquet': RAW_PARQUET.name,
    'source_manifest_sha256': manifest_hash.hexdigest(),   # changes if any source file changes
    'duplicate_max_shift_hours': DUP_MAX_SHIFT_HOURS,
    'minhash': {'shingle_words': SHINGLE_SIZE, 'permutations': MINHASH_PERMUTATIONS, 'seed': RANDOM_SEED, 'text': 'body_clean'},
    'random_seed': RANDOM_SEED,
    'counts': dict(stats),
    'quote_marker_counts': dict(marker_counts),
    'quality_flag_counts': dict(sorted(flag_counts.items())),
    'parse_errors': parse_errors,
    'review_pending': ['quotation extraction per marker type', 'date anomalies',
                       'duplicate copies (identical vs timezone shift)', 'person aliases and custodian coverage',
                       'known-email presence', 'near-duplicate LSH threshold', 'wrongdoing scope and cutoff']
}
(RUN_DIR / 'quality_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')

# Review sample with blank columns for the reviewer to fill in
with (RUN_DIR / 'cleaning_review_sample.jsonl').open('w', encoding='utf-8') as handle:
    for row in sample:
        row.update({'reviewer': '', 'new_text_extraction_correct': None, 'date_checked': None, 'notes': ''})
        handle.write(json.dumps(row, ensure_ascii=False) + '\n')

print(json.dumps({k: v for k, v in report.items() if k != 'parse_errors'}, indent=2))
print('Output directory:', RUN_DIR)
if parse_errors:
    raise RuntimeError(f'Review {len(parse_errors)} parse errors in quality_report.json before using this run.')

{
  "run_name": "20261111T025423476065Z",
  "status": "processing_complete_review_pending",
  "python_version": "3.13.7",
  "pyarrow_version": "26.0.1",
  "numpy_version": "2.5.4",
  "source_url": "https://s3.amazonaws.com/inta.gatech/inta6450-emails.zip",
  "available_json_files": 251734,
  "selected_files": 251734,
  "is_full_corpus": true,
  "raw_parquet": "enron_raw.parquet",
  "source_manifest_sha256": "adcf847d604f764515ec1aa11db66c0ead454dfe97d9b512f77c2e148d55c11b",
  "duplicate_max_shift_hours": 26,
  "minhash": {
    "shingle_words": 5,
    "permutations": 64,
    "seed": 6450,
    "text": "body_clean"
  },
  "random_seed": 6450,
  "counts": {
    "files_attempted": 251734,
    "valid_records": 251734,
    "duplicate_copies": 2491,
    "duplicate_copies_none": 0,
    "unique_content_representatives": 249243,
    "duplicate_copies_timezone_shift": 2491
  },
  "quote_marker_counts": {
    "outlook_original_message": 53419,
    "forwarded_by": 29748,
    "none": 142604,
    "not

## 7. Helpers for later steps
Copy these into EDA and retrieval notebooks (they only need the imports and paths from Section 1).

- `load_raw(email_ids)` returns the **original** parsed JSON (text and headers) for chosen emails, for quoting evidence or checking the cleaning.
- `cutoff_utc('2001-10-16T00:00:00')` converts a Houston local time to UTC with daylight saving applied (midnight Oct 16, 2001 CDT = 05:00 UTC). Filter with `df[df.date_utc < cutoff]`; rows with null `date_utc` are neither before nor after and should be reviewed.

In [12]:
def load_raw(email_ids):
    """Return {email_id: original JSON dict} from enron_raw.parquet for the given IDs."""
    wanted = list(dict.fromkeys(email_ids))
    table = pq.read_table(RAW_PARQUET, columns=['email_id', 'raw_bytes'], filters=[('email_id', 'in', wanted)])
    return {e: json.loads(b.decode('utf-8-sig'))
            for e, b in zip(table.column('email_id').to_pylist(), table.column('raw_bytes').to_pylist())}


def cutoff_utc(houston_local):
    """Houston local time string without an offset, e.g. '2001-10-16T00:00:00' -> UTC datetime."""
    local = datetime.fromisoformat(houston_local)
    if local.tzinfo is not None:
        raise ValueError('Give Houston local time without an offset.')
    return local.replace(tzinfo=HOUSTON).astimezone(timezone.utc)


assert cutoff_utc('2001-10-16T00:00:00') == datetime(2001, 10, 16, 5, tzinfo=timezone.utc)   # CDT, UTC-5
assert cutoff_utc('2001-01-15T00:00:00') == datetime(2001, 1, 15, 6, tzinfo=timezone.utc)    # CST, UTC-6
first_id = pq.read_table(RAW_PARQUET, columns=['email_id']).column('email_id')[0].as_py()
assert 'text' in load_raw([first_id])[first_id]
print('Downstream helpers ready.')

Downstream helpers ready.


## Review and handoff checklist

1. Check `is_full_corpus`, funnel reconciliation, and `parse_errors` in `quality_report.json`. A partial pilot is not a final dataset. An interrupted run has no quality report and must not be used.
2. Inspect the 40-record review sample, plus ~10 examples per `quote_marker_type` (especially `notes_header_block` and the `quote_split_at_start` flag), `duplicate_timezone_shift` copies, and date-flagged records. Use `load_raw` to compare with the originals. Record errors and refine rules before final retrieval.
3. EDA (next notebook) computes monthly volume, custodian/sender coverage, length distributions, term coverage and noise from `emails_analysis.parquet`.
4. Near duplicates: band the `minhash_signature` values for LSH (e.g. 16 bands × 4 rows ≈ 0.5 Jaccard threshold; 8 × 8 ≈ 0.77) and validate the threshold on inspected pairs. Signatures are null for emails with fewer than 5 words.
5. Check named seed emails by sender/date/subject and record their local IDs; do not claim coverage from names alone.
6. Agree on scope and cutoff (Houston time) and apply it with `cutoff_utc`; manually check emails within ~24 hours of the cutoff, since mailbox copies can carry dates shifted by hours.
7. For EDA and retrieval, load selected columns (`pandas.read_parquet(path, columns=[...])`) and keep `is_duplicate_copy == False`. Address columns are always lists, possibly empty: use `len()` checks, and drop NaN rows after `explode()` before building a network. For hosted Elasticsearch, retain its original `_id`, map results to local evidence explicitly, and acknowledge that hosted ranking uses the server's original text, not this cleaned text. Do not modify the shared course index.
8. For each final email, collect all rows with the same `canonical_email_id` to see every mailbox that held a copy (evidence of receipt). Quote from `load_raw`, and record downstream counts (retrieved, reviewed, human-coded, final 5–8 emails). Search scores are not wrongdoing probabilities.